In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import pyreadr
import joblib

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

In [3]:
dem_uncont = pd.read_csv('transformed/dem_uncontested_seats.csv')
rep_uncont = pd.read_csv('transformed/rep_uncontested_seats.csv')

In [4]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont.shape[0])
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims

,seats,winner
0,245,Democrats
1,211,Republicans
2,236,Democrats
3,229,Democrats
4,220,Democrats
...,...,...
19995,227,Democrats
19996,231,Democrats
19997,231,Democrats
19998,236,Democrats


In [5]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 218 else (435-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,150,1,0.005,Republicans
1,151,1,0.005,Republicans
2,152,1,0.005,Republicans
3,158,1,0.005,Republicans
4,160,1,0.005,Republicans


In [6]:
np.unique(seat_sims['seats']).shape[0]

189

In [7]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner', 
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'})
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=217.5, line_width=1, line_color='black', annotation_text='218 seats required for majority', 
                    annotation_position='top right')
sims_hist

In [8]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [9]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,...,19990,19991,19992,19993,19994,19995,19996,19997,19998,19999
AK-AL,-0.672030,-7.356339,-1.556502,-5.773482,-3.094042,-4.370689,-3.891815,-4.858048,-4.784829,-5.323414,...,-6.056714,-1.352019,-5.398098,-6.733619,1.878966,-0.610678,-4.619789,-4.223371,-0.965773,-8.415116
AL-01,-11.956209,-19.090787,-13.330512,-14.614943,-13.116367,-14.078057,-18.586117,-14.900296,-13.095541,-14.961726,...,-16.933476,-8.623670,-14.795062,-18.906752,-13.025948,-13.660733,-17.249143,-12.148882,-12.524740,-13.036306
AL-02,-0.952971,-0.571435,-3.922085,-0.249561,-4.881245,3.515090,-4.950622,-1.576918,0.331831,-4.360302,...,-1.602959,-4.742302,-3.688976,-1.051786,0.763939,0.235478,-1.000408,-0.157234,-2.103721,-0.704788
AL-03,-16.818753,-22.376560,-20.369983,-18.761976,-21.237839,-19.697612,-22.403069,-20.740820,-21.000019,-23.739417,...,-22.102721,-18.289500,-21.622420,-27.401168,-17.629657,-21.377413,-22.033724,-21.460617,-20.656274,-23.865702
AL-04,-26.901761,-33.197737,-26.300172,-28.311586,-31.386670,-28.846433,-30.225058,-29.236476,-28.476340,-32.724453,...,-32.365796,-33.404360,-30.566601,-34.903561,-22.486811,-33.475146,-32.478524,-30.486639,-24.818887,-30.351981


In [28]:
sim_corr = post_untransp.corr()

In [29]:
sim_corr

,AK-AL,AL-01,AL-02,AL-03,AL-04,AL-05,AL-06,AL-07,AR-01,AR-02,AR-03,AR-04,AZ-01,AZ-02,AZ-03,AZ-04,AZ-05,AZ-06,AZ-07,AZ-08,AZ-09,CA-01,CA-02,CA-03,CA-05,CA-06,CA-08,CA-09,CA-10,CA-13,CA-15,CA-16,CA-17,CA-18,CA-19,CA-20,CA-21,CA-22,CA-23,CA-24,CA-25,CA-26,CA-27,CA-28,CA-30,CA-31,CA-32,CA-33,CA-35,CA-36,CA-38,CA-39,CA-41,CA-42,CA-43,CA-44,CA-45,CA-46,CA-47,CA-48,CA-49,CA-50,CA-51,CA-52,CO-01,CO-02,CO-03,CO-04,CO-05,CO-06,CO-07,CO-08,CT-01,CT-02,CT-03,CT-04,CT-05,DE-AL,FL-01,FL-02,FL-03,FL-04,FL-05,FL-06,FL-07,FL-08,FL-09,FL-11,FL-12,FL-13,FL-14,FL-15,FL-16,FL-17,FL-18,FL-19,FL-20,FL-21,FL-22,FL-23,FL-24,FL-25,FL-26,FL-27,FL-28,GA-01,GA-02,GA-03,GA-04,GA-05,GA-06,GA-07,GA-08,GA-09,GA-10,GA-11,GA-12,GA-13,GA-14,HI-01,HI-02,IA-01,IA-02,IA-03,IA-04,ID-01,ID-02,IL-01,IL-02,IL-03,IL-04,IL-05,IL-06,IL-07,IL-08,IL-09,IL-10,IL-11,IL-12,IL-13,IL-14,IL-15,IL-16,IL-17,IN-01,IN-02,IN-03,IN-04,IN-05,IN-06,IN-07,IN-08,IN-09,KS-01,KS-02,KS-03,KS-04,KY-01,KY-02,KY-03,KY-04,KY-05,KY-06,LA-01,LA-02,LA-03,LA-04,LA-05,LA-06,MA-01,MA-02,MA-03,MA-04,MA-05,MA-06,MA-07,MA-08,MA-09,MD-01,MD-02,MD-03,MD-04,MD-05,MD-06,MD-07,MD-08,ME-01,ME-02,MI-01,MI-02,MI-03,MI-04,MI-05,MI-06,MI-07,MI-08,MI-09,MI-10,MI-11,MI-12,MI-13,MN-01,MN-02,MN-03,MN-04,MN-05,MN-06,MN-07,MN-08,MO-01,MO-02,MO-03,MO-04,MO-05,MO-06,MO-07,MO-08,MS-01,MS-02,MS-03,MS-04,MT-01,MT-02,NC-01,NC-02,NC-03,NC-04,NC-05,NC-06,NC-07,NC-08,NC-09,NC-10,NC-11,NC-12,NC-13,NC-14,ND-AL,NE-01,NE-02,NE-03,NH-01,NH-02,NJ-01,NJ-02,NJ-03,NJ-04,NJ-05,NJ-06,NJ-07,NJ-09,NJ-10,NJ-11,NJ-12,NM-01,NM-02,NM-03,NV-01,NV-02,NV-03,NV-04,NY-01,NY-02,NY-03,NY-04,NY-05,NY-06,NY-07,NY-08,NY-09,NY-10,NY-11,NY-12,NY-13,NY-14,NY-15,NY-16,NY-17,NY-18,NY-19,NY-20,NY-21,NY-22,NY-23,NY-24,NY-25,NY-26,OH-01,OH-02,OH-03,OH-04,OH-05,OH-06,OH-07,OH-08,OH-09,OH-10,OH-11,OH-12,OH-13,OH-14,OH-15,OK-01,OK-02,OK-03,OK-04,OK-05,OR-01,OR-02,OR-03,OR-04,OR-05,OR-06,PA-01,PA-02,PA-04,PA-05,PA-06,PA-07,PA-08,PA-09,PA-10,PA-11,PA-12,PA-13,PA-14,PA-15,PA-16,PA-17,RI-01,RI-02,SC-01,SC-02,SC-03,SC-04,SC-05,SC-06,SC-07,SD-AL,TN-01,TN-02,TN-03,TN-04,TN-05,TN-06,TN-07,TN-08,TN-09,TX-01,TX-02,TX-03,TX-04,TX-05,TX-06,TX-07,TX-08,TX-09,TX-10,TX-11,TX-12,TX-13,TX-14,TX-15,TX-16,TX-17,TX-18,TX-19,TX-20,TX-21,TX-22,TX-23,TX-24,TX-25,TX-26,TX-27,TX-28,TX-29,TX-30,TX-31,TX-32,TX-33,TX-34,TX-35,TX-36,TX-37,TX-38,UT-01,UT-02,UT-03,UT-04,VA-01,VA-02,VA-03,VA-04,VA-05,VA-06,VA-07,VA-08,VA-09,VA-10,VA-11,VT-AL,WA-01,WA-02,WA-03,WA-04,WA-05,WA-06,WA-07,WA-08,WA-09,WA-10,WI-01,WI-03,WI-04,WI-05,WI-06,WI-07,WI-08,WV-01,WV-02,WY-AL
AK-AL,1.000000,0.720137,0.504067,0.740507,0.734318,0.723277,0.741214,0.501666,0.739769,0.746811,0.738595,0.743270,0.500329,0.498403,0.503204,0.525595,0.707677,0.724860,0.725031,0.726053,0.744558,0.708238,0.522989,0.526498,0.746198,0.734975,0.583058,0.518386,0.529541,0.526290,0.504175,0.706249,0.581891,0.521484,0.520260,0.710614,0.523060,0.746406,0.734242,0.531602,0.530246,0.710937,0.499724,0.523193,0.500120,0.533349,0.568198,0.527531,0.571324,0.530249,0.712430,0.520463,0.518171,0.506615,0.527880,0.511592,0.495697,0.561385,0.499425,0.710955,0.521035,0.531244,0.516936,0.517708,0.708487,0.520531,0.725702,0.738110,0.725254,0.523012,0.511726,0.724066,0.708193,0.525513,0.523464,0.552485,0.523989,0.491648,0.505474,0.707804,0.740903,0.576201,0.745078,0.710529,0.706849,0.726651,0.521016,0.710481,0.739643,0.743971,0.520184,0.734276,0.705892,0.740781,0.736904,0.707039,0.526843,0.743528,0.707065,0.525135,0.708780,0.510517,0.741079,0.739606,0.562650,0.708051,0.526141,0.556665,0.522107,0.522379,0.519828,0.735000,0.740913,0.740320,0.705346,0.708781,0.740005,0.736042,0.502135,0.515035,0.510181,0.564960,0.710869,0.733772,0.706809,0.561070,0.743738,0.507852,0.710689,0.515971,0.726567,0.574257,0.563472,0.732694,0.712963,0.728583,0.528721,0.527782,0.744323,0.506535,0.525318,0.735980,0.738273,0.508969,0.516565,0.730441,0.732777,0.741822,0.736503,0.515968,0.525761,0.724423,0.733536,0.736264,0.724475,0.517616,0.739396,0.742432,0.734669,0.512212,0.707818,0.736083,0.7

In [11]:
post.shape

(422, 20000)

In [12]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 218 - dem_uncont.shape[0]: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (218 - dem_uncont.shape[0])
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (435 - seats_won_dem) - (218 - rep_uncont.shape[0])
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [13]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|███████████████████████████████████████████████████████████████████████████| 20000/20000 [00:28<00:00, 704.57it/s]


array(['NY-01', 'VA-01', 'MI-04', ..., 'IA-01', 'WI-03', 'OH-15'],
      shape=(20000,), dtype='<U32')

In [14]:
data = pd.read_csv('../../model_output/house_predictions.csv')
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,-6.691602,-18.764506,-1,-37.529012,45.279167,3.612067,9.41,1,38.198416,52.305608
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,-28.655280,-38.923019,0,-77.846039,35.785512,3.594815,0.02,2,28.752683,42.864268
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,-6.824413,-19.716169,1,-39.432338,47.964466,3.519277,27.91,3,40.988201,54.975968
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",...,-39.423181,-45.710827,-1,-91.421653,29.371185,3.505188,0.00,4,22.505018,36.254082
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",...,-59.715277,-47.132870,-1,-94.265741,19.663436,3.553191,0.00,5,12.688429,26.659381


In [15]:
data['tipping_point_prob'] = data['cd'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,-18.764506,-1,-37.529012,45.279167,3.612067,9.41,1,38.198416,52.305608,0.115
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,-38.923019,0,-77.846039,35.785512,3.594815,0.02,2,28.752683,42.864268,0.000
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,-19.716169,1,-39.432338,47.964466,3.519277,27.91,3,40.988201,54.975968,1.395
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",...,-45.710827,-1,-91.421653,29.371185,3.505188,0.00,4,22.505018,36.254082,0.000
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",...,-47.132870,-1,-94.265741,19.663436,3.553191,0.00,5,12.688429,26.659381,0.000


In [16]:
data.sort_values('tipping_point_prob', ascending=False).head(7)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
233,233,NC-11,Jamie Ager,Jennifer Balkcom,False,False,NC,11,"AGER, JAMIE",no_match,...,50.000000,0,100.000000,50.539886,3.702420,56.000,234,43.253319,57.813512,3.220
197,197,MI-10,Christina Hines,Mike Bouchard,False,False,MI,10,"HINES, CHRISTINA","BOUCHARD, MICHAEL",...,-3.195838,0,-6.391676,50.269814,3.684874,52.490,198,43.098468,57.478844,2.945
37,37,CA-22,Randy Villegas,David Valadao,False,True,CA,22,"VILLEGAS, RANDY","VALADAO, DAVID",...,12.732090,-1,25.464181,50.719620,3.506972,57.950,38,43.908238,57.653340,2.800
293,293,OH-07,Brian Poindexter,Max Miller,False,True,OH,7,"POINDEXTER, BRIAN","MILLER, MAX",...,-17.767963,-1,-35.535926,50.365858,3.585064,54.355,294,43.292932,57.346047,2.655
277,277,NY-17,Cait Conley,Mike Lawler,False,True,NY,17,"CONLEY, CAIT","LAWLER, MICHAEL VINCENT",...,3.773526,-1,7.547052,50.376213,3.509991,54.400,278,43.516361,57.242094,2.655
331,331,SC-01,Nancy Lacore,Jenny Costa Honeycutt,False,False,SC,1,"LACORE, NANCY","HONEYCUTT, JENNY COSTA",...,32.983182,0,65.966363,49.154024,3.704446,41.035,332,41.872879,56.540515,2.570
191,191,MI-04,Sean McCann,Bill Huizenga,False,True,MI,4,"MCCANN, SEAN","HUIZENGA, WILLIAM P",...,7.674047,-1,15.348095,49.947175,3.493298,49.510,192,43.100430,56.820153,2.535


In [17]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand):
    indie_d = ['Bill Hill']
    indie_r = ['Kevin Kiley']
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + ' (I)'
        dem_color = 'purple'
    else:
        dem_lab = dem_cand + ' (D)'
        dem_color = 'blue'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + ' (I)'
        rep_color = 'purple'
    else:
        rep_lab = rep_cand + ' (R)'
        rep_color = 'red'

    return f'<p style="color:{dem_color};">' + dem_lab + f'</p> vs <p style="color:{rep_cand}">' + rep_lab + '</p>'

In [18]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand']), axis=1)
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,-37.529012,45.279167,3.612067,9.41,1,38.198416,52.305608,0.115,Very Likely R,"<p style=""color:purple;"">Bill Hill (I)</p> vs ..."
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,-77.846039,35.785512,3.594815,0.02,2,28.752683,42.864268,0.000,Safe R,"<p style=""color:blue;"">Clyde Jones Jr. (D)</p>..."
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,-39.432338,47.964466,3.519277,27.91,3,40.988201,54.975968,1.395,Lean R,"<p style=""color:blue;"">Shomari Figures (D)</p>..."
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",...,-91.421653,29.371185,3.505188,0.00,4,22.505018,36.254082,0.000,Safe R,"<p style=""color:blue;"">Lee McInnis (D)</p> vs ..."
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",...,-94.265741,19.663436,3.553191,0.00,5,12.688429,26.659381,0.000,Safe R,"<p style=""color:blue;"">Amanda Pusczek (D)</p> ..."


In [19]:
data['projected_winner'] = data['chance'].map(lambda x: '(D)' if x > 50 else '(R)')
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,45.279167,3.612067,9.41,1,38.198416,52.305608,0.115,Very Likely R,"<p style=""color:purple;"">Bill Hill (I)</p> vs ...",(R)
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,35.785512,3.594815,0.02,2,28.752683,42.864268,0.000,Safe R,"<p style=""color:blue;"">Clyde Jones Jr. (D)</p>...",(R)
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,47.964466,3.519277,27.91,3,40.988201,54.975968,1.395,Lean R,"<p style=""color:blue;"">Shomari Figures (D)</p>...",(R)


In [20]:
pvi_24 = pd.read_csv('../../transformed/pvi/past_pres_results_by24dist.csv')
data = pd.merge(left=data, right=pvi_24[['district', 'party']], left_on='cd', right_on='district')
data = data.rename({'party': 'curr_party'}, axis=1)
data['hold'] = data['projected_winner'] ==  data['curr_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,52.305608,0.115,Very Likely R,"<p style=""color:purple;"">Bill Hill (I)</p> vs ...",(R),AK-AL,(R),True,False,
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,42.864268,0.000,Safe R,"<p style=""color:blue;"">Clyde Jones Jr. (D)</p>...",(R),AL-01,(R),True,False,


In [21]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic,projected_2p_margin
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,0.115,Very Likely R,"<p style=""color:purple;"">Bill Hill (I)</p> vs ...",(R),AK-AL,(R),True,False,,R+9.4
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,0.000,Safe R,"<p style=""color:blue;"">Clyde Jones Jr. (D)</p>...",(R),AL-01,(R),True,False,,R+28.4
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,1.395,Lean R,"<p style=""color:blue;"">Shomari Figures (D)</p>...",(R),AL-02,(D),False,True,Flip,R+4.1


In [22]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,curr_party,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,(R),True,False,,R+9.4,90.59,9.4,90.6,9.4%,90.6%
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,(R),True,False,,R+28.4,99.98,0.0,100.0,<1%,>99%
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,(D),False,True,Flip,R+4.1,72.09,27.9,72.1,27.9%,72.1%


In [23]:
data['geoid']

0       200
1       101
2       102
3       103
4       104
       ... 
417    5507
418    5508
419    5401
420    5402
421    5600
Name: geoid, Length: 422, dtype: int64

In [24]:
dem_uclen = dem_uncont.shape[0]
dem_uncont['rating'] = np.full(dem_uclen, 'Safe D')
dem_uncont['chance'] = np.full(dem_uclen, 100)
dem_uncont['disp_dem_chance'] = np.full(dem_uclen, '100%')
dem_uncont['disp_rep_chance'] = np.full(dem_uclen, '0%')
dem_uncont['projected_2p_margin'] = np.full(dem_uclen, 'D+100')
dem_uncont['flip_indic'] = np.full(dem_uclen, '')
dem_uncont = dem_uncont.drop(['Unnamed: 0'], axis=1)
dem_uncont['matchup'] = dem_uncont[['dem_cand', 'rep_cand']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand']), axis=1)
dem_uncont.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fips,geoid,rating,chance,disp_dem_chance,disp_rep_chance,projected_2p_margin,flip_indic,matchup
0,CA-04,Mike Thompson/Eric Jones,Not Contested,True,False,CA,4,6,604,Safe D,100,100%,0%,D+100,,"<p style=""color:blue;"">Mike Thompson/Eric Jone..."
1,CA-07,Doris Matsui/Mai Vang,Not Contested,True,False,CA,7,6,607,Safe D,100,100%,0%,D+100,,"<p style=""color:blue;"">Doris Matsui/Mai Vang (..."
2,CA-11,Scott Weiner/Connie Chan,Not Contested,False,False,CA,11,6,611,Safe D,100,100%,0%,D+100,,"<p style=""color:blue;"">Scott Weiner/Connie Cha..."
3,CA-12,Lateefah Simon,Not Contested,True,False,CA,12,6,612,Safe D,100,100%,0%,D+100,,"<p style=""color:blue;"">Lateefah Simon (D)</p> ..."
4,CA-14,Aisha Wahab/Melissa Hernandez,Not Contested,False,False,CA,14,6,614,Safe D,100,100%,0%,D+100,,"<p style=""color:blue;"">Aisha Wahab/Melissa Her..."


In [25]:
rep_uclen = rep_uncont.shape[0]
rep_uncont['rating'] = np.full(rep_uclen, 'Safe R')
rep_uncont['chance'] = np.full(rep_uclen, 0)
rep_uncont['disp_dem_chance'] = np.full(rep_uclen, '0%')
rep_uncont['disp_rep_chance'] = np.full(rep_uclen, '100%')
rep_uncont['projected_2p_margin'] = np.full(rep_uclen, 'R+100')
rep_uncont['flip_indic'] = np.full(rep_uclen, '')
rep_uncont = rep_uncont.drop(['Unnamed: 0'], axis=1)
rep_uncont['matchup'] = rep_uncont[['dem_cand', 'rep_cand']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand']), axis=1)
rep_uncont.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fips,geoid,rating,chance,disp_dem_chance,disp_rep_chance,projected_2p_margin,flip_indic,matchup
0,CA-40,Not Contested,Young Kim/Ken Calvert,False,True,CA,40,6,640,Safe R,0,0%,100%,R+100,,"<p style=""color:blue;"">Not Contested (D)</p> v..."


In [26]:
incl_cols = ['cd', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'flip_indic', 'geoid', 'matchup']
disp_data = pd.concat([data[incl_cols], dem_uncont[incl_cols], rep_uncont[incl_cols]], axis=0)
disp_data.shape

(435, 13)

In [27]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
data.to_csv('display_data/table_display_data.csv')